In [ ]:
"""
Cache a new time-of-day single day grain.
Use that to build all the other aggregations
for multiday.
"""

import datetime
from functools import cache
from typing import Literal, Optional

import pandas as pd
from calitp_data_analysis import utils
from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from dask import compute, delayed
from loguru import logger
from segment_speed_utils import gtfs_schedule_wrangling, helpers, segment_calcs
from segment_speed_utils.project_vars import SEGMENT_TYPES
from shared_utils import portfolio_utils, time_helpers
from update_vars import GTFS_DATA_DICT, SEGMENT_GCS


@cache
def gcs_pandas():
    return GCSPandas()

@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [ ]:
from segment_speed_utils.project_vars import analysis_date_list
segment_type = 'speedmap_segments'
config_path=GTFS_DATA_DICT
analysis_date = analysis_date_list[0]

In [ ]:
dict_inputs = config_path[segment_type]

SPEED_FILE = dict_inputs["stage4"]
MAX_SPEED = dict_inputs["max_speed"]
EXPORT_FILE = dict_inputs["segment_timeofday"]

SEGMENT_COLS = [*dict_inputs["segment_cols"]]
SEGMENT_COLS_NO_GEOM = [i for i in SEGMENT_COLS if i != "geometry"]

OPERATOR_COLS = ["schedule_gtfs_dataset_key"]
CROSSWALK_COLS = [*dict_inputs["crosswalk_cols"]]

group_cols = OPERATOR_COLS + SEGMENT_COLS_NO_GEOM + ["stop_pair_name"] 

df = (
    delayed(gcs_pandas().read_parquet)(
        f"{SEGMENT_GCS}{SPEED_FILE}_{analysis_date}.parquet",
        columns=OPERATOR_COLS
        + SEGMENT_COLS_NO_GEOM
        + ["trip_instance_key", "stop_pair_name", "time_of_day", "speed_mph"],
        filters=[[("speed_mph", "<=", MAX_SPEED)]],
    )
    .dropna(subset="speed_mph")
)

In [ ]:
df_by_time_period = df.pipe(segment_calcs.calculate_avg_speeds, group_cols + ["time_of_day"])
df_all_day = (
    df
    # .drop(columns=['time_of_day'])
    .pipe(segment_calcs.calculate_avg_speeds, group_cols)
    .assign(time_of_day = 'All Day')
)

In [ ]:
# df1 = compute(df_by_time_period)

In [ ]:
# df2 = compute(df_all_day)

In [ ]:
df = delayed(pd.concat)([df_by_time_period, df_all_day], axis=0)

In [ ]:
def merge_schedule_columns_for_speedmaps(df: pd.DataFrame, analysis_date: str, with_all_day: bool):
    """
    Merge in additional columns from scheduled trip table.
    Want route_short_name, as well as n_scheduled_trips and
    trips_per_hour by time-of-day.
    """
    keep_trip_cols = ["trip_instance_key", "gtfs_dataset_key", "route_id", "shape_id", "route_short_name"]

    time_buckets = gtfs_schedule_wrangling.get_trip_time_buckets(analysis_date)

    trips = helpers.import_scheduled_trips(analysis_date, columns=keep_trip_cols, get_pandas=True)
    trips = trips.merge(
        time_buckets, on="trip_instance_key", how="inner"
    )

    route_cols = ["schedule_gtfs_dataset_key", "route_id", "shape_id"]

    # Get trip counts by time-of-day
    schedule_trip_counts = gtfs_schedule_wrangling.count_trips_by_group(
        trips, group_cols=route_cols + ["time_of_day"]
    ).rename(columns={"n_trips": "n_trips_sch"})
    # Add a trips per hour within each time-of-day bucket
    schedule_trip_counts["trips_hr_sch"] = schedule_trip_counts.apply(
        lambda x: round(x.n_trips_sch / time_helpers.HOURS_BY_TIME_OF_DAY[x.time_of_day], 3), axis=1
    )

    if with_all_day:
        schedule_trip_counts_all_day = gtfs_schedule_wrangling.count_trips_by_group(
        trips, group_cols=route_cols
            ).rename(columns={"n_trips": "n_trips_sch"}).assign(time_of_day = 'All Day')
        schedule_trip_counts_all_day["trips_hr_sch"] = schedule_trip_counts.apply(
        lambda x: round(x.n_trips_sch / 24, 3), axis=1 # use 24 hours for now
        )
        schedule_trip_counts = pd.concat([schedule_trip_counts, schedule_trip_counts_all_day])

    df2 = pd.merge(df, schedule_trip_counts, on=route_cols + ["time_of_day"], how="inner").merge(
        trips[route_cols + ["route_short_name"]].drop_duplicates(), on=route_cols, how="inner"
    )

    return df2

In [ ]:
if segment_type == "speedmap_segments":
    df = delayed(merge_schedule_columns_for_speedmaps)(df, analysis_date, with_all_day=True).pipe(
        gtfs_schedule_wrangling.merge_operator_identifiers, [analysis_date], columns=CROSSWALK_COLS
    )

    # segment_id should capture the 1,000m segments, where it's suffixed -1, -2, -3
    SEGMENT_COLS = [
        "shape_array_key",  # exclude shape_id
        "route_id",
        "direction_id",
        "stop_pair",
        "segment_id",
        "geometry",
    ]

avg_speeds_with_geom = delayed(segment_calcs.merge_in_segment_geometry)(
    df, [analysis_date], segment_type, SEGMENT_COLS
)

avg_speeds_with_geom = compute(avg_speeds_with_geom)[0]
avg_speeds_with_geom = portfolio_utils.standardize_operator_info_for_exports(avg_speeds_with_geom, analysis_date)
drop_cols = [
    col for col in avg_speeds_with_geom.columns if "original" in col or "organization" in col or "org" in col
]
avg_speeds_with_geom = avg_speeds_with_geom.drop(columns=drop_cols).drop_duplicates()

In [ ]:
avg_speeds_with_geom.time_of_day.value_counts()

In [ ]:
print(f'{SEGMENT_GCS}{EXPORT_FILE}_{analysis_date}.parquet')
gcs_geopandas().geo_data_frame_to_parquet(avg_speeds_with_geom, f'{SEGMENT_GCS}{EXPORT_FILE}_{analysis_date}.parquet')